# LAB | First match ≠ right match
## Relevance Scoring & Rerankers for Trustworthy AI + EU AI Act

**Learning goal:** build one retrieval path over two sources and **prove** every answer is grounded in evidence you can inspect.

**Your sources**
- `podcast.txt`: a podcast transcript about the EU *Ethics Guidelines for Trustworthy AI* (2019)
- `eu_ai_act.pdf`: Regulation (EU) 2024/1689, the AI Act (144 pages)

**What you submit:** a `lab_proof.md` containing your code (or a link to this notebook), **one query**, the **retrieved evidence**, the **final answer**, and **one failure or limitation**. There's a template at the end of this notebook.

---

### How this notebook works

| Label | Meaning |
|---|---|
| **PROVIDED** | Code is given. Run it and read it. |
| **YOUR TURN** | Replace the `# TODO` lines with your own code. |
| **OPTIONAL (Advanced)** | Do this if you have time; it's not required to pass. |

Each step ends with a **Done when** check. Don't move on until it passes.

---
## Step 0: Environment [PROVIDED]

1. Create a `.env` file next to this notebook containing `OPENAI_API_KEY=...` and `PINECONE_API_KEY=...`
2. Create a folder called `data/` next to this notebook and put `podcast.txt` and `eu_ai_act.pdf` inside it.
3. Run the two cells below.

**Warning:** Never paste API keys into a notebook cell, and never commit your `.env` file.

In [9]:
%pip uninstall -y -q pinecone-client
%pip install -U -q --force-reinstall pinecone
%pip install -U -q python-dotenv pypdf langchain-core langchain-community langchain-text-splitters langchain-openai langchain-pinecone

Note: you may need to restart the kernel to use updated packages.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-pinecone 0.2.13 requires pinecone[asyncio]<8.0.0,>=6.0.0, but you have pinecone 10.2.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

for key in ["OPENAI_API_KEY", "PINECONE_API_KEY"]:
    if not os.getenv(key):
        raise ValueError(f"{key} not found. Add it to your .env file.")
print("API keys loaded")

API keys loaded


---
## Step 1: Setup & Data Preparation [PROVIDED, except 1d]

**Goal:** turn two very different sources into chunks that each carry **metadata**, meaning labels that say *where the chunk came from*.

**Why metadata matters:** when the system answers *"Fines are up to €35M"*, a reviewer must be able to trace that sentence back to *"EU AI Act, page 115"*. Without metadata you can't trace an answer, and you can't filter results either (Step 5).

Parts 1a–1c are done for you: run each cell and **read the output**. Part 1d (embeddings) is yours to write.

### 1a. Load both sources

In [2]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, TextLoader

DATA_DIR = Path("data")
PODCAST_PATH = DATA_DIR / "podcast.txt"
PDF_PATH = DATA_DIR / "eu_ai_act.pdf"

for path in [PODCAST_PATH, PDF_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Put the lab files in a 'data' folder next to this notebook.")

podcast_docs = TextLoader(str(PODCAST_PATH), encoding="utf-8").load()  # 1 Document (whole transcript)
pdf_docs = PyPDFLoader(str(PDF_PATH)).load()                           # 1 Document per PDF page

print(f"Podcast: {len(podcast_docs)} document, {len(podcast_docs[0].page_content):,} characters")
print(f"EU AI Act: {len(pdf_docs)} pages")
print("\nRaw metadata from the PDF loader:", pdf_docs[0].metadata)

/var/folders/44/wy0yrhws1v1b2q2kl0jqk4bc0000gn/T/ipykernel_68688/320955083.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader


Podcast: 1 document, 16,414 characters
EU AI Act: 144 pages

Raw metadata from the PDF loader: {'producer': 'PDFlib+PDI 9.0.7p3 (C++/Win64)', 'creator': 'Servigistics Arbortext Publishing Engine', 'creationdate': '2024-07-11T14:47:17+02:00', 'author': 'Publications Office of the European Union L-2985 Luxembourg LUXEMBOURG', 'keywords': 'ISSN 1977-0677', 'moddate': '2024-07-11T15:55:28+02:00', 'subject': 'I Legislative acts', 'title': 'Regulation (EU) 2024/1689 of the European Parliament and of the Council of 13 June 2024 laying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008, (EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and Directives 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Artificial Intelligence Act)Text with EEA relevance.', 'source': 'data/eu_ai_act.pdf', 'total_pages': 144, 'page': 0, 'page_label': '1'}


### 1b. Chunk with metadata

In [3]:
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

def update_regulation_location(text, section, article):
    """Track the structural heading that applies to the current chunk."""
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if re.fullmatch(r"ANNEX\s+[IVXLCDM]+\b.*", line, flags=re.IGNORECASE):
            section, article = "annex", None
        elif re.fullmatch(r"CHAPTER\s+[IVXLCDM]+\b.*", line, flags=re.IGNORECASE):
            section, article = "article", None
        else:
            match = re.fullmatch(r"Article\s+(\d+)\b.*", line, flags=re.IGNORECASE)
            if match and section == "article":
                article = f"Article {match.group(1)}"
    return section, article

def chunk_with_metadata(docs, source_type, source_name):
    """Split documents and attach traceable, filterable metadata."""
    chunks = []
    section = "podcast" if source_type == "podcast" else "recital"
    article = None
    chunk_number = 0

    # Process one source document at a time so article headings carry across page boundaries.
    for doc in docs:
        for chunk in splitter.split_documents([doc]):
            if source_type == "regulation":
                section, article = update_regulation_location(chunk.page_content, section, article)

            metadata = {
                "source_type": source_type,
                "source_name": source_name,
                "chunk_id": f"{source_type}-{chunk_number}",
                "section": section,
            }
            # PDF pages are 0-indexed in the loader; store the human-readable page number.
            if "page" in chunk.metadata:
                metadata["page"] = int(chunk.metadata["page"]) + 1
            # Pinecone metadata cannot contain None, so omit article when it is not applicable.
            if article is not None and section == "article":
                metadata["article"] = article

            chunk.metadata = metadata
            chunks.append(chunk)
            chunk_number += 1
    return chunks

podcast_chunks = chunk_with_metadata(podcast_docs, "podcast", "Trustworthy AI podcast")
act_chunks = chunk_with_metadata(pdf_docs, "regulation", "EU AI Act (Reg. 2024/1689)")
all_chunks = podcast_chunks + act_chunks

print(f"Podcast chunks:   {len(podcast_chunks)}")
print(f"EU AI Act chunks: {len(act_chunks)}")
print(f"Total chunks:     {len(all_chunks)}")

Podcast chunks:   20
EU AI Act chunks: 764
Total chunks:     784


### 1c. Inspect your chunks (always look at your data!)

In [4]:
for chunk in [podcast_chunks[0], act_chunks[150]]:
    print(chunk.metadata)
    print(chunk.page_content[:300], "...")
    print("-" * 80)

{'source_type': 'podcast', 'source_name': 'Trustworthy AI podcast', 'chunk_id': 'podcast-0'}
So imagine for a second you're driving across a, I don't know, a massive suspension bridge. Okay. You don't pull over halfway across, get out and demand to see the blueprints, right? You don't interview the welding crew. No, you just, you trust it. You just drive, you trust the bridge, you trust the ...
--------------------------------------------------------------------------------
{'source_type': 'regulation', 'source_name': 'EU AI Act (Reg. 2024/1689)', 'chunk_id': 'regulation-150', 'page': 21}
indicators relating to AI.
OJ L, 12.7.2024 EN
ELI: http://data.europa.eu/eli/reg/2024/1689/oj 21/144
(35) Directive 2014/31/EU of the European Parliament and of the Council of 26 February 2014 on the harmonisation of the laws of the 
Member States relating to the making available on the market of no ...
--------------------------------------------------------------------------------


### 1d. Embeddings model [YOUR TURN]

Set up the embedding model yourself and test it on **one** chunk.

**What to do**
1. Import `OpenAIEmbeddings` from `langchain_openai`.
2. Create a variable called `embeddings` using the model `text-embedding-3-small`. Step 2 uses this exact variable name.
3. Embed the text of `all_chunks[0]` with `embeddings.embed_query(...)` and print the length of the vector.

**Hint:** see **Section 2 of `rag_intro.ipynb`**.
**Note:** you don't embed **all** chunks here. That happens in Step 2, when you upload them to Pinecone, so you only pay for them once.

In [5]:
# YOUR TURN: set up the embedding model and test it on one chunk
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
test_vector = embeddings.embed_query(all_chunks[0].page_content)
print(f"Embedding length: {len(test_vector)}")

Embedding length: 1536


### Stop and think (write your answers in `lab_proof.md` later)

1. Our metadata has `source_type`, `source_name`, `chunk_id` and `page`. **What's missing?** Think about a user asking *"What does Article 5 say?"* or *"Only show me recitals."* Could you filter on that today?
2. The podcast has no pages. How would a reviewer find where a podcast chunk came from?

You'll come back to this in **Step 5**.

**Done when:** you can see chunk counts for both sources, every chunk you print shows a `metadata` dict, and your test embedding has length **1536**.

---
## Step 2: Embeddings + Baseline Retrieval [YOUR TURN]

**Goal:** build a plain vector search. This is your **baseline**: every later step must beat it, or it isn't worth the extra cost.

**What to do**
1. Create (or connect to) a Pinecone index. Use dimension `1536` and metric `cosine`.
2. Upload `all_chunks` with `PineconeVectorStore.from_documents(...)`. This is the step that embeds every chunk.
3. Write `baseline_search(query, k)`. It should return `(doc, score)` pairs using `similarity_search_with_score`.
4. Run it on the **fixed test queries** below and save the results.

**Hint:** Stuck? Look at **Section 3–4 of `rag_intro.ipynb`** (yesterday's notebook).
**Warning:** Only run the upload cell **once**. Running it again uploads duplicate chunks.

In [6]:
# PROVIDED: fixed test queries. Use the SAME queries in every step so you can compare fairly.
TEST_QUERIES = [
    "What AI practices are prohibited under the EU AI Act?",
    "What are the seven requirements for trustworthy AI?",
    "What is the difference between human-in-the-loop and human-on-the-loop?",
    "What fines apply for using a prohibited AI practice?",
    "Is social scoring of citizens allowed?",
]

# PROVIDED: helper to print results in the same format in every step
def show_results(query, results, label="Results"):
    """results: list of (Document, score) pairs"""
    print(f"\n=== {label} | Query: {query}")
    for rank, (doc, score) in enumerate(results, 1):
        m = doc.metadata
        where = f"p.{m['page']}" if "page" in m else m.get("chunk_id")
        print(f"#{rank}  score={score:.3f}  [{m.get('source_type')} | {where}]  {doc.page_content[:120]!r}")

In [7]:
# YOUR TURN: create / connect to the Pinecone index
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index_name = "lab-rerankers"   # use lowercase letters and dashes only

if index_name not in [index.name for index in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f"Created index: {index_name}")
else:
    print(f"Index {index_name} already exists")

vectorstore = PineconeVectorStore.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    index_name=index_name
)

print("Documents added to Pinecone vector store")

/opt/miniconda3/envs/bootcamp-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Created index: lab-rerankers
Documents added to Pinecone vector store


In [8]:
# YOUR TURN: baseline retrieval
def baseline_search(query, k=5):
    return vectorstore.similarity_search_with_score(query, k=k)

baseline_results = {}
for q in TEST_QUERIES:
    results = baseline_search(q, k=5)
    baseline_results[q] = results
    show_results(q, results, label="Baseline")


=== Baseline | Query: What AI practices are prohibited under the EU AI Act?
#1  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#2  score=0.684  [regulation | p.12.0]  '(45) Practices that are prohibited by Union law, including data protection law, non-discrimination law, consumer \nprotec'
#3  score=0.681  [regulation | p.9.0]  'intention to distort behaviour where the distortion results from factors external to the AI system which are outside \nth'
#4  score=0.673  [regulation | p.45.0]  '(b) prohibitions of certain AI practices;\n(c) specific requirements for high-risk AI systems and obligations for operato'
#5  score=0.660  [regulation | p.3.0]  'operators involved in the AI value chain under this Regulation should apply without prejudice to national law, in \ncompl'

=== Baseline | Query: What are the seven requirements for trustworthy AI?
#1  score=0.674  [podcast | p

**Done when:** each test query prints 5 results with a score, a source type and a page or chunk id.

**Note it down:** for which query does the **top result** look wrong or only half-relevant? That's your *first match ≠ right match* case.

---
## Step 3: LLM Relevance Scoring [OPTIONAL (Advanced)]

**Goal:** have an LLM judge how relevant each retrieved chunk is, then re-order the results.

**What to do**
1. Retrieve a wider set of candidates (e.g. `k=10`) with `baseline_search`.
2. For each chunk, ask `gpt-4o-mini` to score its relevance to the query from 0 to 1.
3. Combine the two scores, e.g. `combined = 0.3 * similarity + 0.7 * llm_score`, and sort by `combined`.
4. Compare the new top 3 with the baseline top 3.

**Hint:** Reuse `rerank_documents()` from **Section 3 of `rag_pipeline.ipynb`**. It already parses the score and prints it.
**Warning:** This makes 1 LLM call **per chunk, per query**. Count the calls you make.

In [ ]:
# YOUR TURN
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def llm_relevance_score(query, doc):
    # TODO: prompt the LLM for a 0-1 score, parse it robustly, return a float
    pass

def llm_rerank(query, k_initial=10, k_final=5, w_sim=0.3, w_llm=0.7):
    # TODO: retrieve k_initial -> score each chunk -> combine scores -> sort -> return top k_final as (doc, combined_score)
    pass

**Done when:** you can show at least one query where the order changed, and say whether it got better or worse.

---
## Step 4: Dedicated Reranker (Cross-Encoder or Cohere) [OPTIONAL (Advanced)]

**Goal:** use a model built specifically for reranking. It's usually faster and cheaper than an LLM, and often better.

**Option A: local cross-encoder (no API key needed)**
```
%pip install -q sentence-transformers
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
scores = reranker.predict([(query, doc.page_content) for doc in docs])
```

**Option B: Cohere Rerank API** (needs `COHERE_API_KEY` in `.env`; see the Cohere Rerank docs)

**What to do:** rerank the same 10 baseline candidates, keep the top 5, and compare with Step 3.

In [13]:
# YOUR TURN
from sentence_transformers import CrossEncoder

# This model runs locally after its first download.
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def crossencoder_rerank(query, k_initial=10, k_final=5):
    """Retrieve candidates, score query/chunk pairs, and keep the best chunks."""
    initial_results = baseline_search(query, k=k_initial)
    documents = [doc for doc, _ in initial_results]
    pairs = [(query, doc.page_content) for doc in documents]
    scores = reranker.predict(pairs)

    reranked = sorted(
        [(doc, float(score)) for doc, score in zip(documents, scores)],
        key=lambda item: item[1],
        reverse=True,
    )
    return reranked[:k_final]

W1028 15:36:57.594000 68688 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 21449.54it/s]


**Done when:** you have a side-by-side of baseline vs reranked top 3 for at least two queries.

---
## Step 5: Metadata Filtering [YOUR TURN]

**Goal:** restrict search to the right part of the knowledge base *before* ranking.

**What to do**
1. Filter by source using Pinecone's filter syntax:
   `vectorstore.similarity_search_with_score(query, k=5, filter={"source_type": {"$eq": "regulation"}})`
2. Run one query twice: once with **only the regulation**, once with **only the podcast**. How do the answers differ?
3. **The real lesson:** `section` (`"recital"` / `"article"` / `"annex"`) and `article` (`"Article 5"`) were not present in the original metadata. They are now added in Step 1, so rerun Step 1b and use the next cell to **delete the old index and re-upload** before filtering on them.

**Hint:** Hints for the EU AI Act: in this PDF the recitals run up to about page 44, the Articles start at *"CHAPTER I"*, and the Annexes start at *"ANNEX I"*. A regex like `r"^Article\s+(\d+)"` on each line can detect article headings.
**Warning:** Pinecone metadata values can't be `None`. Leave the key out instead.

In [9]:
# YOUR TURN: filter by source_type
query = TEST_QUERIES[0]

regulation_results = vectorstore.similarity_search_with_score(
    query,
    k=5,
    filter={"source_type": {"$eq": "regulation"}},
)
podcast_results = vectorstore.similarity_search_with_score(
    query,
    k=5,
    filter={"source_type": {"$eq": "podcast"}},
)

show_results(query, regulation_results, label="Regulation only")
show_results(query, podcast_results, label="Podcast only")


=== Regulation only | Query: What AI practices are prohibited under the EU AI Act?
#1  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#2  score=0.684  [regulation | p.12.0]  '(45) Practices that are prohibited by Union law, including data protection law, non-discrimination law, consumer \nprotec'
#3  score=0.682  [regulation | p.9.0]  'intention to distort behaviour where the distortion results from factors external to the AI system which are outside \nth'
#4  score=0.673  [regulation | p.45.0]  '(b) prohibitions of certain AI practices;\n(c) specific requirements for high-risk AI systems and obligations for operato'
#5  score=0.660  [regulation | p.3.0]  'operators involved in the AI value chain under this Regulation should apply without prejudice to national law, in \ncompl'

=== Podcast only | Query: What AI practices are prohibited under the EU AI Act?
#1  score=0.519 

In [10]:
# YOUR TURN: filter by the NEW metadata field you added (section / article / ...)
# Run this cell once, after rerunning Step 1b so all_chunks has section/article metadata.
# This replaces the old index because Pinecone does not retroactively update metadata.
import time

if index_name in [index.name for index in pc.list_indexes()]:
    pc.delete_index(index_name)
    print(f"Deleted old index: {index_name}")
    while index_name in [index.name for index in pc.list_indexes()]:
        time.sleep(2)

pc.create_index(
    name=index_name,
    dimension=1536,
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)

vectorstore = PineconeVectorStore.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    index_name=index_name
)
print("Rebuilt Pinecone index with section/article metadata")

article_5_results = vectorstore.similarity_search_with_score(
    "What AI practices are prohibited?",
    k=5,
    filter={
        "section": {"$eq": "article"},
        "article": {"$eq": "Article 5"},
    },
)
show_results("What AI practices are prohibited?", article_5_results, label="Article 5 only")

Deleted old index: lab-rerankers
Rebuilt Pinecone index with section/article metadata

=== Article 5 only | Query: What AI practices are prohibited?


**Done when:** you can run a query that only returns, e.g., *Article* chunks from the EU AI Act, and you can explain why you had to rebuild the index to do it.

---
## Step 6: Complete RAG Pipeline with Reranking [YOUR TURN]

**Goal:** plug your best retrieval method into a full RAG chain that **cites its sources**.

**What to do**
1. Build an LCEL chain: `retrieve → format context → prompt → LLM → answer` (see **Section 5 of `rag_intro.ipynb`**).
2. In `format_context`, put each chunk's source and page **inside** the context, e.g. `[EU AI Act, p.51]`, and tell the LLM to cite them.
3. Make it easy to switch retrieval methods: baseline vs. reranked should be a **1–2 line change**.
4. Make the model say *"I don't know based on the sources"* when the context doesn't contain the answer.

In [11]:
# YOUR TURN
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def format_context(results):
    """Format retrieved (Document, score) pairs as traceable context."""
    context_blocks = []
    for doc, score in results:
        metadata = doc.metadata
        source = metadata["source_name"]
        location = (
            f"p.{int(metadata['page'])}"
            if "page" in metadata
            else metadata["chunk_id"]
        )
        citation = f"[{source}, {location}]"
        context_blocks.append(
            f"{citation}\n{doc.page_content}"
        )
    return "\n\n---\n\n".join(context_blocks)

rag_prompt = ChatPromptTemplate.from_template(
    """Answer the question using only the supplied context.

Every factual claim must include a citation copied exactly from the
context, such as [EU AI Act (Reg. 2024/1689), p.51] or
[Trustworthy AI podcast, podcast-8]. If the context does not support
the answer, say: \"I don't know based on the sources.\"

Context:
{context}

Question: {question}

Answer:"""
)

rag_chain = rag_prompt | llm | StrOutputParser()

def answer(query, retrieve_fn=baseline_search):
    """Retrieve, answer from formatted context, and return the evidence used."""
    results = retrieve_fn(query)
    response = rag_chain.invoke({
        "context": format_context(results),
        "question": query,
    })
    return response, results

query = TEST_QUERIES[3]
final_answer, answer_evidence = answer(query)
print(f"Question: {query}\n")
print(final_answer)
show_results(query, answer_evidence, label="Evidence used for final answer")

Question: What fines apply for using a prohibited AI practice?

Non-compliance with the prohibition of the AI practices referred to in Article 5 shall be subject to administrative fines of up to EUR 35 000 000 or, if the offender is an undertaking, up to 7 % of its total worldwide annual turnover for the preceding financial year, whichever is higher [EU AI Act (Reg. 2024/1689), p.115].

=== Evidence used for final answer | Query: What fines apply for using a prohibited AI practice?
#1  score=0.633  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#2  score=0.617  [regulation | p.118.0]  '(d) failed to make available to the Commission access to the general-purpose AI model or general-purpose AI model with \n'
#3  score=0.613  [regulation | p.115.0]  'including start-ups, and their economic viability.\n2. The Member States shall, without delay and at the latest by the da'
#4  score=0.610  [

**Done when:** one query gives an answer with citations, and you can point to the exact chunk behind each claim.

---
## Step 7: Evaluate: With vs. Without Reranking [YOUR TURN]

**Goal:** decide, with evidence, whether reranking was worth it.

**What to do:** for each test query, read the answers and the retrieved chunks yourself, then fill in the table. You are the judge: no automatic metric.

| Query | Method | Top-1 chunk relevant? (Y/N) | Answer correct? (Y/Partial/N) | Grounded in cited chunk? (Y/N) | Notes |
|---|---|---|---|---|---|
| Q1 | Baseline | | | | |
| Q1 | Reranked | | | | |
| Q2 | Baseline | | | | |
| Q2 | Reranked | | | | |
| … | | | | | |

**Then answer:**
- Did reranking change the answer, or only the order?
- What did it cost (extra calls, seconds per query)? Would a client pay for that?

In [14]:
# YOUR TURN (optional helper): print baseline vs reranked answers side by side, and time each one
import time

evaluation_results = {}

for q in TEST_QUERIES:
    baseline_start = time.perf_counter()
    baseline_answer, baseline_evidence = answer(q, baseline_search)
    baseline_seconds = time.perf_counter() - baseline_start

    reranked_start = time.perf_counter()
    reranked_answer, reranked_evidence = answer(q, crossencoder_rerank)
    reranked_seconds = time.perf_counter() - reranked_start

    evaluation_results[q] = {
        "baseline": {
            "answer": baseline_answer,
            "evidence": baseline_evidence,
            "seconds": baseline_seconds,
        },
        "reranked": {
            "answer": reranked_answer,
            "evidence": reranked_evidence,
            "seconds": reranked_seconds,
        },
    }

    print(f"\n{'=' * 80}\nQUERY: {q}")
    print(f"\nBASELINE ({baseline_seconds:.2f}s):\n{baseline_answer}")
    show_results(q, baseline_evidence[:3], label="Baseline top 3")
    print(f"\nRERANKED ({reranked_seconds:.2f}s):\n{reranked_answer}")
    show_results(q, reranked_evidence[:3], label="Reranked top 3")

print("\nManual evaluation table — fill in the judgment columns after inspecting the answers and evidence:")
print("| Query | Method | Top-1 relevant? | Answer correct? | Grounded? | Notes |")
print("|---|---|---|---|---|---|")
for q in TEST_QUERIES:
    print(f"| {q} | Baseline | | | | |")
    print(f"| {q} | Reranked | | | | |")


QUERY: What AI practices are prohibited under the EU AI Act?

BASELINE (2.61s):
The following AI practices are prohibited under the EU AI Act: "the placing on the market, the putting into service or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective, or the effect of materially distorting the behaviour of a person or a group of persons by appreciably impairing their ability to make an intention to distort behaviour" [EU AI Act (Reg. 2024/1689), p.51].

=== Baseline top 3 | Query: What AI practices are prohibited under the EU AI Act?
#1  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#2  score=0.684  [regulation | p.12.0]  '(45) Practices that are prohibited by Union law, including data protection law, non-discrimination law, consumer \nprotec'
#3  score=0.682  [

---
## Step 8: Write `lab_proof.md` [YOUR TURN]

Create `lab_proof.md` in your repo (this is the **entry point for your reviewer**). Use this template:

```markdown
# Lab Proof: First match ≠ right match

## 1. What I built
One or two sentences, plus a link to this notebook.

## 2. Query
"..."

## 3. Retrieved evidence
| Rank | Source | Page / chunk_id | Score | Snippet |
|---|---|---|---|---|

## 4. Final answer
(paste the answer, including its citations)

## 5. Trace
Which claim in the answer comes from which chunk?

## 6. Baseline vs reranked
What changed? Was it worth the cost?

## 7. Failure or limitation
Where could this system give a plausible but unsupported answer?
```

**Before you submit:**
- [ ] Only files for this lab are in the repo
- [ ] No API keys, `.env` or tokens are committed
- [ ] `lab_proof.md` is the entry point
- [ ] A reviewer can trace **one** answer back to the exact retrieved chunk

---
### Extensions (optional)
- **Hybrid search:** combine BM25 keyword search with vector search, then rerank
- **Cost analysis:** cost per query for baseline vs LLM scoring vs cross-encoder
- **Failure test:** ask something neither source covers. Does your pipeline say "I don't know"?